# Database Secrets Engine: Lumen RO/RW

`./demo.sh` incluye la creación del catálogo, tablas y roles. Se reutiliza MySQL/KMIP del [notebook 2](2_vault-demo-kmip-mysql.ipynb). La conexión y los roles de la aplicación están en el namespace Vault **verify**, no en el namespace raíz.

| Rol | Privilegios SQL | TTL / máximo |
|---|---|---|
| `chatbot-readonly` | SELECT sobre `scripting.*` | 120 s / 300 s |
| `chatbot-readwrite` | SELECT, INSERT, UPDATE, DELETE sobre `scripting.*` | 120 s / 300 s |

Vault administra la cuenta `vault_chatbot_admin` y rota su contraseña al crear la conexión. La aplicación solicita credenciales con OBO y RAR; el perfil humano, el límite del agente y el RAR deben permitir la ruta.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/01-database.sh


In [ ]:
%%bash
source verify/scripts/common.sh
for role in chatbot-readonly chatbot-readwrite; do
  vault read -format=json "database/roles/$role" | jq --arg role "$role" '{role:$role,db_name:.data.db_name,default_ttl:.data.default_ttl,max_ttl:.data.max_ttl,creation_statements:.data.creation_statements}'
done


## Verificación

Después del despliegue completo, la comprobación valida SELECT, permite INSERT con RW, deniega INSERT con RO y hace rollback. Comprueba además Tavily y los límites del token de bootstrap. Las pruebas OBO por usuario requieren su login real en Verify.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/test-infrastructure.sh


Las antiguas conexiones `database/config/mysql` y el static role `payments-app` del namespace raíz pertenecen a otro ejercicio; si existen se conservan, pero Lumen no los utiliza. No se incluye la aplicación `5_Payments_App.ipynb`, que no forma parte de este workspace.

El [notebook 8](8_Lumen_Protected_Applications.ipynb) añade Transit/Transform. El supervisor usa SQL RO y permisos separados para revelar datos; el gestor usa SQL RW y puede proteger datos, sin revelarlos.
